In [ ]:
import numpy as np
import os
import cv2
from PIL import Image
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
import matplotlib.pyplot as plt

In [ ]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define the path to your dataset in Google Drive
image_directory = '/content/drive/My Drive/cell_images/'

In [ ]:
# Set the image size
SIZE = 64
dataset = []
label = []

In [ ]:
# Iterate through all images in the Parasitized folder
parasitized_images = os.listdir(image_directory + 'parasitized/')
for i, image_name in enumerate(parasitized_images):
    if image_name.endswith('.png'):
        image = cv2.imread(image_directory + 'parasitized/' + image_name)
        image = Image.fromarray(image, 'RGB')
        image = image.resize((SIZE, SIZE))
        dataset.append(np.array(image))
        label.append(0)

In [ ]:
# Iterate through all images in the Uninfected folder
uninfected_images = os.listdir(image_directory + 'uninfected/')
for i, image_name in enumerate(uninfected_images):
    if image_name.endswith('.png'):
        image = cv2.imread(image_directory + 'uninfected/' + image_name)
        image = Image.fromarray(image, 'RGB')
        image = image.resize((SIZE, SIZE))
        dataset.append(np.array(image))
        label.append(1)

In [ ]:
INPUT_SHAPE = (SIZE, SIZE, 3)
inp = keras.layers.Input(shape=INPUT_SHAPE)
conv1 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same')(inp)
pool1 = keras.layers.MaxPooling2D(pool_size=(2, 2))(conv1)
norm1 = keras.layers.BatchNormalization()(pool1)
drop1 = keras.layers.Dropout(rate=0.2)(norm1)
conv2 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(drop1)  # Increased filters
pool2 = keras.layers.MaxPooling2D(pool_size=(2, 2))(conv2)
norm2 = keras.layers.BatchNormalization()(pool2)
drop2 = keras.layers.Dropout(rate=0.2)(norm2)

flat = keras.layers.Flatten()(drop2)
hidden1 = keras.layers.Dense(512, activation='relu')(flat)
norm3 = keras.layers.BatchNormalization()(hidden1)
drop3 = keras.layers.Dropout(rate=0.2)(norm3)
hidden2 = keras.layers.Dense(256, activation='relu')(drop3)
norm4 = keras.layers.BatchNormalization()(hidden2)
drop4 = keras.layers.Dropout(rate=0.2)(norm4)

out = keras.layers.Dense(2, activation='softmax')(drop4)  # Change to softmax for categorical classification

model = keras.Model(inputs=inp, outputs=out)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(model.summary())

In [ ]:
# Split the dataset into training and testing datasets
X_train, X_test, y_train, y_test = train_test_split(dataset, to_categorical(np.array(label)), test_size=0.20, random_state=0)

In [ ]:

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
# Fit the generator on the training data
datagen.fit(X_train)

In [ ]:
# Train the model
history = model.fit(np.array(X_train),
                    y_train,
                    batch_size=64,
                    verbose=1,
                                     epochs=50,  # Set to 5 for testing; you can change this as needed
                    validation_split=0.1,
                    shuffle=False)

In [ ]:
# Calculate accuracy on the test data
print("Test Accuracy: {:.2f}%".format(model.evaluate(np.array(X_test), np.array(y_test))[1] * 100))

# Plot training history
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
t = f.suptitle('CNN Performance', fontsize=12)
f.subplots_adjust(top=0.85, wspace=0.3)
max_epoch = len(history.history['accuracy']) + 1
epoch_list = list(range(1, max_epoch))
ax1.plot(epoch_list, history.history['accuracy'], label='Train Accuracy')
ax1.plot(epoch_list, history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_xticks(np.arange(1, max_epoch, 1))
ax1.set_ylabel('Accuracy Value')
ax1.set_xlabel('Epoch')
ax1.set_title('Accuracy')
l1 = ax1.legend(loc="best")

ax2.plot(epoch_list, history.history['loss'], label='Train Loss')
ax2.plot(epoch_list, history.history['val_loss'], label='Validation Loss')
ax2.set_xticks(np.arange(1, max_epoch, 1))
ax2.set_ylabel('Loss Value')
ax2.set_xlabel('Epoch')
ax2.set_title('Loss')
l2 = ax2.legend(loc="best")

# Save the model
model.save('/content/drive/My Drive/malaria_cnn.h5')


In [ ]:
from google.colab import files
from PIL import Image
import numpy as np
import cv2

# Upload an image
uploaded = files.upload()

In [ ]:
# Load the uploaded image
for filename in uploaded.keys():
    # Open the image
    img = Image.open(filename)

    # Resize the image to (64, 64)
    img = img.resize((64, 64))

    # Convert the image to a numpy array and scale to [0, 1]
    img_array = np.array(img) / 255.0

    # Expand dimensions to match the input shape of the model
    img_array = np.expand_dims(img_array, axis=0)  # Shape: (1, 64, 64, 3)

In [ ]:
# Load the model
from keras.models import load_model
model = load_model('/content/drive/My Drive/malaria_cnn.h5')  # Ensure you have saved the model earlier

# Make prediction
prediction = model.predict(img_array)

In [ ]:
# Convert predictions to class labels (0 for parasitized, 1 for uninfected)
class_label = np.argmax(prediction, axis=1)
print(class_label)

if class_label[0] == 0:
    print("The image is of a parasitized cell.")
else:
    print("The image is of an uninfected cell.")